# ViT seq2seq — walkthrough

This notebook runs the happy path end to end against the real trained checkpoint. It imports the actual model class and config from the `vit-seq2seq` service rather than reimplementing anything inline.

The generation step is the interesting one: `generate` tracks completion per sequence within a batch, so shorter and longer numbers each run to their own EOS.


In [ ]:
import os

# This notebook lives at notebooks/vit-seq2seq/; the code it exercises lives at
# services/vit-seq2seq/. chdir there so the relative paths in Config
# ("checkpoints/...", "data") resolve exactly as they do for `make serve` and the
# trainers, which run from the service directory. Every workspace member is installed
# by `make install`, so no import-path manipulation is needed.
os.chdir(os.path.join("..", "..", "services", "vit-seq2seq"))

import torch
from ai_residency_common import get_device, load_checkpoint

from config import Config

config = Config()
device = get_device()

print(f"Using device: {device}")
print(f"Classifier arch: {config.classifier_arch}")
print(f"Seq2seq arch:    {config.seq2seq_arch}")
print(f"Classifier checkpoint: {config.classifier_checkpoint_path}")
print(f"Seq2seq checkpoint:    {config.seq2seq_checkpoint_path}")

## 2. Seq2seq multi-digit recognition

Load the real `Seq2SeqVisionTransformer`, restore the committed checkpoint (84x84 input, patch
size 16, `d_model=256`), composite real multi-digit images with `MultiDigitMNISTDataset` (the
same on-the-fly compositor the trainer uses) and call the model's own `.generate()` — the same
autoregressive decoding path the API's `predict_seq2seq()` takes.


In [ ]:
from vit_seq2seq.seq2seq import Seq2SeqVisionTransformer

seq2seq = Seq2SeqVisionTransformer(config.seq2seq_arch)
load_checkpoint(config.seq2seq_checkpoint_path, seq2seq)
seq2seq.to(device).eval()

total_params = sum(p.numel() for p in seq2seq.parameters())
print(f"Loaded seq2seq model from {config.seq2seq_checkpoint_path} ({total_params:,} params)")

In [ ]:
from datasets.multi_digit_dataset import MultiDigitMNISTDataset

# samples_per_class=1 keeps this notebook fast — a handful of real composited 3-digit images
# demonstrates the trained checkpoint working. The trainer's own evaluate() does the full sweep.
multi_digit = MultiDigitMNISTDataset(
    data_dir=config.data_dir,
    image_size=config.seq2seq_arch.img_size,
    max_digits=config.max_digits,
    samples_per_class=1,
    batch_size=6,
    num_workers=0,
)
_, _, seq_test_loader = multi_digit.get_data_loaders()

seq_images, seq_numbers = next(iter(seq_test_loader))
seq_images = seq_images.to(device)

with torch.no_grad():
    generated = seq2seq.generate(
        seq_images,
        start_token=config.seq2seq_arch.start_token,
        max_length=config.seq2seq_arch.max_seq_length,
    )


def decode_sequence(token_row):
    """Read a generated token row back as the digits it spells."""
    digits = []
    for token in token_row.tolist():
        if token == config.seq2seq_arch.start_token:
            continue
        if token == config.seq2seq_arch.eos_token:
            break
        digits.append(str(token))
    return "".join(digits) if digits else "0"


for i in range(len(seq_numbers)):
    true_number = str(int(seq_numbers[i].item())).zfill(config.max_digits)
    print(f"true={true_number}  generated={decode_sequence(generated[i])}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(seq_numbers), figsize=(2.5 * len(seq_numbers), 2.5))
for i, ax in enumerate(axes):
    true_number = str(int(seq_numbers[i].item())).zfill(config.max_digits)
    ax.imshow(seq_images[i, 0].cpu(), cmap="gray")
    ax.set_title(f"true {true_number} / gen {decode_sequence(generated[i])}")
    ax.axis("off")
plt.tight_layout()
plt.show()